# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MotazSameh/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Build the feature vector

I use five features from the March 2026 decision period. These features describe search visibility and available engagement signals that can be observed before the prediction moment.

The feature vector contains GSC impressions, GSC clicks, average position, GA4 pageviews, and GA4 sessions. Missing values are filled with 0 because unavailable measurements are represented as missing in the warehouse.


In [2]:
from huggingface_hub import hf_hub_download
import pandas as pd

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

df_march = pd.read_parquet(march_file)

print("Shape:", df_march.shape)
print("\nColumns:")
print(df_march.columns.tolist())

print("\nDate range:")
print(df_march["report_date"].min(), "to", df_march["report_date"].max())

print("\nFirst 5 rows:")
display(df_march.head())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Shape: (9841378, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Date range:
2026-03-01 to 2026-03-31

First 5 rows:


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ============================================================
# 1. Build the feature vector
# ============================================================
import numpy as np
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

# Keep only the selected features
feature_vector = df_march[feature_cols].copy()

# Replace infinite values
feature_vector = feature_vector.replace(
    [np.inf, -np.inf],
    np.nan
)

# Handle missing values
feature_vector = feature_vector.fillna(0)

print("Feature vector shape:", feature_vector.shape)

print("\nFeatures:")
print(feature_vector.columns.tolist())

print("\nMissing values after filling:")
print(feature_vector.isna().sum())

print("\nFirst 5 rows:")
display(feature_vector.head())

Feature vector shape: (9841378, 5)

Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions']

Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_pageviews       0
ga4_sessions        0
dtype: int64

First 5 rows:


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,20,0,3.350000,0.0,0.0
1,1,0,0.000000,0.0,0.0
2,125,1,4.928000,0.0,0.0
3,7,0,4.000000,0.0,0.0
4,11,0,2.272727,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)


* **gsc_impressions** — the number of Google Search impressions for the content. Missing values are filled with 0. It is available during the March decision period.
* **gsc_clicks** — the number of Google Search clicks. Missing values are filled with 0. It is available during the March decision period.
* **gsc_avg_position** — the average search position observed for the content. Missing values are filled with 0. It is available during the March decision period.
* **ga4_pageviews** — pageviews recorded by GA4. Missing values are filled with 0. It is available when GA4 data is available during the decision period.
* **ga4_sessions** — sessions recorded by GA4. Missing values are filled with 0. It is available when GA4 data is available during the decision period.

All five features are intended to represent information available before the future outcome is observed.


In [5]:
# Check missing values before filling

missing_before = df_march[feature_cols].isna().sum()

print("Missing values before filling:")
print(missing_before)

print("\nMissing values after filling:")
print(feature_vector.isna().sum())

Missing values before filling:
gsc_impressions           0
gsc_clicks                0
gsc_avg_position    6230317
ga4_pageviews       3018741
ga4_sessions        3018741
dtype: int64

Missing values after filling:
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
ga4_pageviews       0
ga4_sessions        0
dtype: int64


### 3. The leakage hunt

I checked the selected features for information that would only become available after the March decision point.

The final feature vector contains only March-period signals. I deliberately exclude April performance and any future outcome-derived fields. April impressions were used earlier as a deliberate leakage example, but they are not part of the final feature vector.


In [6]:
# ============================================================
# 3. Leakage hunt
# ============================================================

future_or_leaky_features = [
    "april_impressions",
    "future_decline",
    "trend_pct",
    "trend_direction"
]

leaks_present = [
    col for col in future_or_leaky_features
    if col in feature_vector.columns
]

print("Potential leakage columns found in feature vector:")
print(leaks_present)

if len(leaks_present) == 0:
    print("\nLeakage check: PASS")
    print("No known future/label-derived columns are included.")
else:
    print("\nLeakage check: FAIL")
    print("Remove these columns before modeling.")

Potential leakage columns found in feature vector:
[]

Leakage check: PASS
No known future/label-derived columns are included.


### 4. What I excluded and why

* **April performance metrics** — excluded because they are future information relative to the March decision moment.
* **Future decline / outcome columns** — excluded because they directly describe the outcome being predicted.
* **trend_direction** — excluded because it represents the future outcome and would leak the label.
* **trend_pct** — excluded because it is derived from the performance change used to define the outcome.
* **Product decision flags or health scores** — excluded because they may already contain a previous decision or recommendation.
* **Client identifiers and content identifiers** — excluded as model features because they identify entities rather than provide a meaningful pre-decision signal.


In [7]:
excluded_fields = [
    "april_impressions",
    "future_decline",
    "trend_direction",
    "trend_pct",
    "client_hash_id",
    "content_hash_id"
]

print("Excluded fields:")
for field in excluded_fields:
    print("-", field)

Excluded fields:
- april_impressions
- future_decline
- trend_direction
- trend_pct
- client_hash_id
- content_hash_id


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.